# Semana 3: Práctica. DCF de una empresa real

**Curso:** Tópicos de Finanzas Avanzadas (ECON-421, UPAO 2026-20)

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JonathanRosasV/topicos-finanzas-upao/blob/main/03_dcf_fcff_fcfe/clase03_practica.ipynb)

Hoy valoramos Southern Copper (SCCO) por flujo de caja descontado, de principio a fin: FCFF histórico por dos rutas, proyección a 5 años, valor terminal, y el puente hasta el valor por acción. Usamos el WACC que estimamos la semana pasada.

**Requisito previo:** haber corrido `git pull` en tu fork para tener `utils/finanzas.py` actualizado.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt

from utils.finanzas import (capm, wacc, fcff_desde_ebit, fcff_desde_cfo,
                            fcfe_desde_fcff, valor_crecimiento_constante)

TICKER = "SCCO"
tk = yf.Ticker(TICKER)
print("Empresa:", tk.info.get("longName", TICKER))

## 1. Estados financieros

`yfinance` entrega tres estados: resultados (`income_stmt`), balance (`balance_sheet`) y flujo de efectivo (`cashflow`), con los años en columnas (el más reciente primero).

**Ojo con el orden:** al juntar series de estados distintos en una sola tabla, `pandas` reordena las fechas de la más antigua a la más reciente. Por eso ordenamos la tabla de forma explícita, para que `.iloc[0]` sea siempre el año más reciente en todo el notebook.

Los nombres de las filas pueden variar entre empresas, así que usamos una función defensiva que busca la primera etiqueta disponible.

In [ ]:
def fila(df, *nombres):
    """Devuelve la primera fila disponible entre los nombres candidatos."""
    for n in nombres:
        if n in df.index:
            return df.loc[n]
    raise KeyError(f"Ninguna de estas filas existe: {nombres}")

est = tk.income_stmt
bal = tk.balance_sheet
cf  = tk.cashflow

ebit    = fila(est, "EBIT", "Operating Income")
interes = fila(est, "Interest Expense").abs()
impuesto = fila(est, "Tax Provision")
util_at  = fila(est, "Pretax Income")
dep     = fila(cf, "Depreciation And Amortization", "Depreciation Amortization Depletion", "Depreciation")
capex   = fila(cf, "Capital Expenditure").abs()
d_wc    = -fila(cf, "Change In Working Capital")   # signo: incremento de WC consume caja
cfo     = fila(cf, "Operating Cash Flow")

t_efectiva = float((impuesto / util_at).iloc[0])
print(f"Tasa efectiva de impuestos (ultimo anio): {t_efectiva:.1%}")

resumen = pd.DataFrame({"EBIT": ebit, "Dep": dep, "Capex": capex,
                        "WCInv": d_wc, "Interes": interes, "CFO": cfo}).dropna() / 1e6
resumen = resumen.sort_index(ascending=False)   # el anio mas reciente primero: .iloc[0] = ultimo anio
resumen.round(0)

## 2. FCFF histórico por dos rutas

Calculamos el FCFF de cada año desde el EBIT y desde el CFO. Si las dos rutas no se parecen, algo está mal en los insumos (esa es la gracia del control).

In [ ]:
anios = resumen.index
fcff_ebit = pd.Series(
    [fcff_desde_ebit(resumen.loc[a, "EBIT"], t_efectiva, resumen.loc[a, "Dep"],
                     resumen.loc[a, "Capex"], resumen.loc[a, "WCInv"]) for a in anios],
    index=anios, name="FCFF (ruta EBIT)")

fcff_cfo = pd.Series(
    [fcff_desde_cfo(resumen.loc[a, "CFO"], resumen.loc[a, "Interes"],
                    t_efectiva, resumen.loc[a, "Capex"]) for a in anios],
    index=anios, name="FCFF (ruta CFO)")

comparacion = pd.concat([fcff_ebit, fcff_cfo], axis=1)
comparacion["diferencia %"] = (fcff_ebit / fcff_cfo - 1) * 100
comparacion.round(1)

**Pregunta de discusión:** las dos rutas rara vez coinciden al centavo con datos publicados. ¿Qué partidas explican la diferencia? (Pista: la ruta CFO ya trae otros cargos que no son caja además de la depreciación, impuestos diferidos por ejemplo, y la ruta EBIT usa una tasa efectiva promedio.)

## 3. El WACC de la semana pasada

Reconstruimos el WACC con los mismos supuestos declarados de la práctica 2: beta ajustado de la regresión, tasa libre de riesgo del Treasury a 10 años y ERP de 5.5 por ciento.

In [ ]:
# Supuestos declarados (los de la practica de la semana 2; puedes actualizarlos)
rf, ERP, beta_aj = 0.045, 0.055, None   # beta_aj se recalcula abajo

import statsmodels.api as sm
px = yf.download([TICKER, "^GSPC"], start="2021-08-01",
                 interval="1mo", auto_adjust=True, progress=False)["Close"]
r = px.pct_change().dropna().rename(columns={"^GSPC": "SP500"})
modelo = sm.OLS(r[TICKER], sm.add_constant(r["SP500"]), missing="drop").fit()
beta_aj = 0.67 * float(modelo.params["SP500"]) + 0.33

ke = capm(rf, beta_aj, ERP)

E = tk.fast_info["marketCap"] / 1e6
D = float(fila(bal, "Total Debt").iloc[0]) / 1e6
kd = float(resumen["Interes"].iloc[0]) / D
t_marginal = 0.30    # aproximacion para SCCO: opera sobre todo en Peru (29.5%) y Mexico (30%)

WACC = wacc(E, D, ke, kd, t_marginal)
print(f"beta ajustado = {beta_aj:.3f} | Ke = {ke:.2%} | Kd = {kd:.2%}")
print(f"E = {E:,.0f} MM | D = {D:,.0f} MM | WACC = {WACC:.2%}")
print(f"Control: {kd*(1-t_marginal):.2%} < {WACC:.2%} < {ke:.2%}")

## 4. Proyección del FCFF a 5 años

Partimos del FCFF más reciente (ruta EBIT) y declaramos supuestos de crecimiento: más alto al inicio, convergiendo a un crecimiento maduro de largo plazo. La regla de oro: el crecimiento perpetuo no puede superar el crecimiento nominal de la economía.

In [ ]:
fcff_0 = float(fcff_ebit.iloc[0])          # ultimo anio disponible
print(f"Anio base: {fcff_ebit.index[0].year} | FCFF base = {fcff_0:,.0f} MM")
g_proyeccion = [0.06, 0.055, 0.05, 0.045, 0.04]   # supuestos declarados
g_perpetuo = 0.03

proy = []
f = fcff_0
for g in g_proyeccion:
    f = f * (1 + g)
    proy.append(f)

proy = pd.Series(proy, index=range(1, 6), name="FCFF proyectado")
print(proy.round(0))

## 5. Valor terminal y valor presente

El valor terminal captura todo lo que pasa después del año 5, con la perpetuidad creciente (el detalle fino queda para la semana 4). Descontamos todo al WACC.

In [ ]:
VT5 = valor_crecimiento_constante(proy.iloc[-1] * (1 + g_perpetuo), WACC, g_perpetuo)

vp_flujos = sum(cf_i / (1 + WACC) ** i for i, cf_i in proy.items())
vp_VT = VT5 / (1 + WACC) ** 5
EV = vp_flujos + vp_VT

print(f"VP de los 5 flujos      = {vp_flujos:,.0f} MM")
print(f"VP del valor terminal   = {vp_VT:,.0f} MM")
print(f"EV                      = {EV:,.0f} MM")
print(f"Peso del valor terminal = {vp_VT/EV:.0%}   (fijate que grande es)")

## 6. El puente: de EV al valor por acción

In [ ]:
caja = float(fila(bal, "Cash And Cash Equivalents",
                  "Cash Cash Equivalents And Short Term Investments").iloc[0]) / 1e6
deuda_neta = D - caja
acciones = tk.fast_info["shares"] / 1e6

eq = EV - deuda_neta
precio_modelo = eq / acciones
precio_mercado = tk.fast_info["lastPrice"]

print(f"EV {EV:,.0f} - deuda neta {deuda_neta:,.0f} = equity {eq:,.0f} MM")
print(f"Valor por accion (modelo) = {precio_modelo:,.2f}")
print(f"Precio de mercado         = {precio_mercado:,.2f}")
print(f"Diferencia                = {precio_modelo/precio_mercado - 1:+.1%}")

**Interpretación con cuidado profesional:** antes de gritar compra o venta, recuerda la semana 1 (la diferencia mezcla mispricing y error de estimación) y mira el peso del valor terminal: la mayor parte de tu valor descansa en los supuestos de perpetuidad, que es exactamente lo que trabajaremos la semana 4 con análisis de sensibilidad.

## 7. Gráfico: de dónde viene el valor

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
partes = [vp_flujos, vp_VT]
ax.bar(["VP flujos (5 anios)", "VP valor terminal"], partes, color=["#000080", "#31859C"])
for i, v in enumerate(partes):
    ax.text(i, v, f"{v:,.0f}", ha="center", va="bottom")
ax.set_title(f"{TICKER}: composicion del EV ({vp_VT/EV:.0%} es valor terminal)")
ax.set_ylabel("millones de USD")
plt.tight_layout(); plt.show()

## 8. DCF inverso: ¿qué está descontando el mercado?

Cuando el modelo y el precio difieren mucho, la pregunta profesional no es "¿quién tiene razón?" sino "¿qué tendría que creer para pagar este precio?". El DCF inverso le da la vuelta al modelo: deja fijos los flujos proyectados y el WACC, y busca el crecimiento perpetuo que iguala el valor por acción del modelo con el precio de mercado.

Si ese crecimiento implícito supera el crecimiento nominal de largo plazo de la economía (la regla de oro de la sección 4), hay tres lecturas posibles: el mercado es demasiado optimista, nuestra tasa de descuento es demasiado alta, o nuestro flujo base es demasiado bajo. Decidir entre las tres es el trabajo del analista.

In [ ]:
def valor_por_accion(g_perp):
    """Valor por accion del modelo para un crecimiento perpetuo dado."""
    vt = valor_crecimiento_constante(proy.iloc[-1] * (1 + g_perp), WACC, g_perp)
    ev = vp_flujos + vt / (1 + WACC) ** 5
    return (ev - deuda_neta) / acciones

# Biseccion: el valor por accion crece con g, buscamos el g que iguala el precio
lo, hi = -0.05, WACC - 0.0005
if valor_por_accion(hi) < precio_mercado:
    print("Ni con g pegado al WACC el modelo alcanza el precio: revisar flujo base y tasa.")
else:
    for _ in range(60):
        mid = (lo + hi) / 2
        if valor_por_accion(mid) < precio_mercado:
            lo = mid
        else:
            hi = mid
    g_implicito = (lo + hi) / 2
    print(f"g perpetuo que usamos       = {g_perpetuo:.2%}")
    print(f"g perpetuo implicito en {precio_mercado:,.2f} = {g_implicito:.2%}")
    print(f"WACC                        = {WACC:.2%}")

**El flujo base en una empresa cíclica.** SCCO vende cobre, y su FCFF sube y baja con el precio del metal. Proyectar desde un solo año supone que ese año es representativo. Una alternativa prudente es normalizar: usar como base el promedio de los últimos años en lugar del último.

In [ ]:
fcff_promedio = float(fcff_ebit.mean())
print(f"FCFF ultimo anio      = {fcff_0:,.0f} MM")
print(f"FCFF promedio {len(fcff_ebit)} anios = {fcff_promedio:,.0f} MM")
print(f"Si el modelo partiera del promedio, el EV cambiaria en {fcff_promedio / fcff_0 - 1:+.0%} (el DCF es proporcional al flujo base)")

**Pregunta de discusión:** ¿el crecimiento implícito te parece defendible para una minera de cobre? ¿Qué supuesto cambiarías primero: el flujo base, la tasa o el crecimiento? La semana 4 lo formalizamos con tablas de sensibilidad.

## 9. Cierre

Lo que queda en la librería del curso desde hoy: `fcff_desde_ebit()`, `fcff_desde_ni()`, `fcff_desde_cfo()`, `fcfe_desde_fcff()` y `valor_crecimiento_constante()` en `utils/finanzas.py`.

**Tarea de la semana** (`clase03_tarea.ipynb`): replicar este DCF con la empresa que elegiste en la semana 2. Entrega hasta el lunes de la semana siguiente, vía commit en tu fork.

**Próxima semana:** valor terminal y enterprise value, más análisis de sensibilidad formal.